In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
import re
import string
import nltk
import joblib

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:

# ==========================================
# DOWNLOAD NLTK DATA
# (Runs only first time)
# ==========================================

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\malot\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:

# ==========================================
# STEMMER + STOPWORDS
# ==========================================

ps = PorterStemmer()

stop_words = set(stopwords.words('english'))


In [4]:
# ==========================================
# TEXT CLEANING FUNCTION
# ==========================================

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Remove spaces
    text = text.strip()

    # Tokenization
    words = text.split()

    # Remove stopwords + stemming
    words = [
        ps.stem(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)


In [6]:
# ==========================================
# LOAD DATASET
# ==========================================

print("Loading dataset...")

df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="latin-1"
)

print("\nFirst 5 rows:")
print(df.head())

Loading dataset...

First 5 rows:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [7]:

# ==========================================
# CONVERT LABELS
# ham=0 spam=1
# ==========================================

df['label'] = df['label'].map(
    {
        'ham':0,
        'spam':1
    }
)

print("\nDataset Shape:")
print(df.shape)



Dataset Shape:
(5572, 2)


In [8]:

# ==========================================
# PREPROCESS DATA
# ==========================================

print("\nCleaning messages...")

df['message'] = df['message'].apply(
    clean_text
)


Cleaning messages...


In [9]:
# ==========================================
# SPLIT DATA
# ==========================================

X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [10]:
# ==========================================
# BUILD MODEL
# ==========================================

model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svm', LinearSVC())
])



In [11]:
# ==========================================
# TRAIN MODEL
# ==========================================

print("\nTraining model...")

model.fit(
    X_train,
    y_train
)

print("Training completed")



Training model...
Training completed


In [12]:
# ==========================================
# PREDICTIONS
# ==========================================

y_pred = model.predict(X_test)



In [13]:
# ==========================================
# EVALUATION
# ==========================================

print("\nAccuracy Score:")
print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


Accuracy Score:
0.9865470852017937

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.91      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115


Confusion Matrix:
[[964   2]
 [ 13 136]]


In [14]:
# ==========================================
# SAVE MODEL
# ==========================================

joblib.dump(
    model,
    "spam_detector_model.pkl"
)

print("\nModel Saved Successfully")




Model Saved Successfully


In [15]:
# ==========================================
# CUSTOM MESSAGE TESTING
# ==========================================

sample_message = [

    "Congratulations! You won a free iPhone. Click here now",

    "Hi Preethi, are we meeting today at 5 PM?",

    "URGENT! Claim your cash reward now"

]


prediction = model.predict(
    sample_message
)


print("\n========= RESULTS =========")

for msg, pred in zip(
    sample_message,
    prediction
):

    print("\nMessage:")
    print(msg)

    if pred == 1:
        print("Prediction: SPAM")

    else:
        print("Prediction: NOT SPAM")


========= RESULTS =========

Message:
Congratulations! You won a free iPhone. Click here now
Prediction: NOT SPAM

Message:
Hi Preethi, are we meeting today at 5 PM?
Prediction: NOT SPAM

Message:
URGENT! Claim your cash reward now
Prediction: SPAM
